In [88]:
import pandas as pd
import sys
import matplotlib.pyplot as plt
import sklearn
from sklearn.impute import KNNImputer
import altair as alt
alt.data_transformers.enable("vegafusion")

from sklearn.pipeline import Pipeline, make_pipeline

In [89]:
df = pd.read_csv('C:/Users/J.Heuvelmans/OneDrive - Brain Research Center/Documenten/EAISI/2024Supermarket/Code/data/processed/downcasted_history_per_year.csv', header=0)

C:\Users\J.Heuvelmans\AppData\Local\Temp\ipykernel_11368\1303377615.py:1: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('C:/Users/J.Heuvelmans/OneDrive - Brain Research Center/Documenten/EAISI/2024Supermarket/Code/data/processed/downcasted_history_per_year.csv', header=0)


In [90]:
df.head()

,id,store_nbr,item_nbr,unit_sales,onpromotion,date
0,0,25,103665,7.0,NaN,2013-01-01
1,1,25,105574,1.0,NaN,2013-01-01
2,2,25,105575,2.0,NaN,2013-01-01
3,3,25,108079,1.0,NaN,2013-01-01
4,4,25,108701,1.0,NaN,2013-01-01


In [91]:
df = df.drop(columns=['id', 'onpromotion'])
df['date'] = pd.to_datetime(df['date'])

In [92]:
# Define the date range you want to ensure is covered
start_date = '2013-01-01'
end_date = '2017-08-15'
complete_date_range = pd.date_range(start=start_date, end=end_date)

# Create a DataFrame with all combinations of store_nbr and complete_date_range
stores = df['store_nbr'].unique()

df_dates = pd.DataFrame({
    'date': pd.concat([pd.Series(complete_date_range)] * len(stores), ignore_index=True),
    'store_nbr': sorted(list(stores) * len(complete_date_range))
})

df_dates['date'] = pd.to_datetime(df_dates['date'])

df_dates

,date,store_nbr
0,2013-01-01,1
1,2013-01-02,1
2,2013-01-03,1
3,2013-01-04,1
4,2013-01-05,1
...,...,...
91147,2017-08-11,54
91148,2017-08-12,54
91149,2017-08-13,54
91150,2017-08-14,54


In [93]:
def select_item_store(item, store, df):
    df_item = df[df['item_nbr'] == item]
    df_merged = pd.merge(df_dates, df_item, on=['store_nbr', 'date'], how='left')
    df_item_store = df_merged[df_merged['store_nbr'] == store]
    df_item_store['item_nbr'] = df_item_store['item_nbr'].fillna(item)
    return df_item_store

In [204]:
item = 105574
store = 25
df_selection = select_item_store(item, store, df)
df_selection

C:\Users\J.Heuvelmans\AppData\Local\Temp\ipykernel_11368\452726492.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_item_store['item_nbr'] = df_item_store['item_nbr'].fillna(item)


,date,store_nbr,item_nbr,unit_sales
40512,2013-01-01,25,105574.0,1.0
40513,2013-01-02,25,105574.0,5.0
40514,2013-01-03,25,105574.0,2.0
40515,2013-01-04,25,105574.0,5.0
40516,2013-01-05,25,105574.0,5.0
...,...,...,...,...
42195,2017-08-11,25,105574.0,4.0
42196,2017-08-12,25,105574.0,4.0
42197,2017-08-13,25,105574.0,4.0
42198,2017-08-14,25,105574.0,NaN


In [214]:
# Apply the rolling mean imputation method
df_rolling = df_selection.copy()
imputed_indices = df_rolling[df_rolling['unit_sales'].isnull()].index
df_rolling['unit_sales'] = df_rolling['unit_sales'].fillna(df_rolling['unit_sales'].rolling(window=7, min_periods=1).mean().shift(1))

# Apply the forward fill method
df_locf = df_selection.fillna(method="ffill")

# Apply the mean imputation method
df_mean = df_selection.fillna(df['unit_sales'].mean())

# Apply the KNN imputer
imputer = KNNImputer(n_neighbors=10)
df_knn = df_selection.copy()
df_knn[['unit_sales']] = imputer.fit_transform(df_knn[['unit_sales']])

C:\Users\J.Heuvelmans\AppData\Local\Temp\ipykernel_11368\3085009070.py:7: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_locf = df_selection.fillna(method="ffill")


In [215]:
df_chart = df_selection

chart1 = alt.Chart(df_chart).mark_line(opacity=1, color = 'blue').encode(
    x='date:T',
    y='unit_sales:Q'
).properties(
    title='Unit Sales over time per store',
    width=8000,
    height=400
)

In [216]:
df_chart = df_rolling

chart2 = alt.Chart(df_chart).mark_line(opacity=0.5, color = 'red').encode(
    x='date:T',
    y='unit_sales:Q'
).properties(
    title='Unit Sales over time per store',
    width=8000,
    height=400
)

In [217]:
df_chart = df_locf

chart3 = alt.Chart(df_chart).mark_line(opacity=0.5, color = 'red').encode(
    x='date:T',
    y='unit_sales:Q'
).properties(
    title='Unit Sales over time per store',
    width=8000,
    height=400
)

In [218]:
df_chart = df_mean

chart4 = alt.Chart(df_chart).mark_line(opacity=0.5, color = 'yellow').encode(
    x='date:T',
    y='unit_sales:Q'
).properties(
    title='Unit Sales over time per store',
    width=8000,
    height=400
)

In [219]:
df_chart = df_knn

chart5 = alt.Chart(df_chart).mark_line(opacity=0.5, color = 'yellow').encode(
    x='date:T',
    y='unit_sales:Q'
).properties(
    title='Unit Sales over time per store',
    width=8000,
    height=400
)

In [220]:
# Overlay the two charts
combined_chart = chart1 + chart2 + chart5
combined_chart

alt.LayerChart(...)